In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

In [3]:
class Lenet_5(nn.Module):
  def __init__(self):
    super(Lenet_5,self).__init__()
    self.fe = nn.Sequential(
        nn.Conv2d(in_channels = 1, out_channels = 6, kernel_size = 5, stride = 1, padding = 2),
        nn.ReLU(),
        nn.AvgPool2d(kernel_size = 2, stride = 2),
        nn.Conv2d(in_channels = 6, out_channels = 16, kernel_size = 5, stride = 1, padding = 0),
        nn.ReLU(),
        nn.AvgPool2d(kernel_size = 2, stride = 2)
    )

    self.classifier = nn.Sequential(
        nn.Linear(in_features = 400, out_features = 120),
        nn.ReLU(),
        nn.Linear(in_features = 120, out_features = 84),
        nn.ReLU(),
        nn.Linear(in_features = 84, out_features = 10)
    )

  def forward(self,x):
    x = self.fe(x)
    x = x.view(x.shape[0],-1)
    x = self.classifier(x)
    return x



In [8]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,)),

])

train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loaders = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loaders = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

100%|██████████████████████████████████████| 9.91M/9.91M [00:16<00:00, 618kB/s]
100%|█████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 93.6kB/s]
100%|██████████████████████████████████████| 1.65M/1.65M [00:03<00:00, 512kB/s]
100%|█████████████████████████████████████| 4.54k/4.54k [00:00<00:00, 1.12MB/s]


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Lenet_5().to(device)

print(model)

Lenet_5(
  (fe): Sequential(
    (0): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
  )
  (classifier): Sequential(
    (0): Linear(in_features=400, out_features=120, bias=True)
    (1): ReLU()
    (2): Linear(in_features=120, out_features=84, bias=True)
    (3): ReLU()
    (4): Linear(in_features=84, out_features=10, bias=True)
  )
)


In [6]:
# hyper-parameters

num_epochs = 10
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr = 0.001)

In [11]:
def train_loop(model, train_loader, criterion, optimizer):
  model.train()

  running_loss = 0
  correct = 0
  total = 0

  for images,labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    # Prediction
    logits = model(images)

    # Compute the loss
    loss = criterion(logits,labels)

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()

    # Weight update
    optimizer.step()

    running_loss += loss.item()

    predictions = torch.argmax(logits,dim = 1)
    correct += (predictions == labels).sum().item()
    total += labels.size(0)

  epoch_loss = running_loss / len(train_loader)
  epoch_acc = correct / total

  print(f"Train Loss : {epoch_loss:.4f} | Train Acc : {epoch_acc:.4f}")



for epoch in range(num_epochs):
  print(f"Epoch : {epoch+1}")
  train_loop(model,train_loaders,criterion,optimizer)


Epoch : 1
Train Loss : 0.4481 | Train Acc : 0.8609
Epoch : 2
Train Loss : 0.1153 | Train Acc : 0.9649
Epoch : 3
Train Loss : 0.0784 | Train Acc : 0.9755
Epoch : 4
Train Loss : 0.0635 | Train Acc : 0.9800
Epoch : 5
Train Loss : 0.0531 | Train Acc : 0.9837
Epoch : 6
Train Loss : 0.0456 | Train Acc : 0.9852
Epoch : 7
Train Loss : 0.0392 | Train Acc : 0.9878
Epoch : 8
Train Loss : 0.0359 | Train Acc : 0.9885
Epoch : 9
Train Loss : 0.0320 | Train Acc : 0.9895
Epoch : 10
Train Loss : 0.0286 | Train Acc : 0.9905


In [10]:
import math
math.floor(3.66)

3